##### random forest regression

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('cardekho_imputated.csv')

In [3]:
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


##### finding null values 

In [4]:
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [5]:

print(df.describe(),"\n")
print(df.dtypes)

df.head()


         Unnamed: 0   vehicle_age     km_driven       mileage        engine  \
count  15411.000000  15411.000000  1.541100e+04  15411.000000  15411.000000   
mean    9811.857699      6.036338  5.561648e+04     19.701151   1486.057751   
std     5643.418542      3.013291  5.161855e+04      4.171265    521.106696   
min        0.000000      0.000000  1.000000e+02      4.000000    793.000000   
25%     4906.500000      4.000000  3.000000e+04     17.000000   1197.000000   
50%     9872.000000      6.000000  5.000000e+04     19.670000   1248.000000   
75%    14668.500000      8.000000  7.000000e+04     22.700000   1582.000000   
max    19543.000000     29.000000  3.800000e+06     33.540000   6592.000000   

          max_power         seats  selling_price  
count  15411.000000  15411.000000   1.541100e+04  
mean     100.588254      5.325482   7.749711e+05  
std       42.972979      0.807628   8.941284e+05  
min       38.400000      0.000000   4.000000e+04  
25%       74.000000      5.000000

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [6]:
### brand, model is not necessary as it is already give in the car_name

df=df.drop(columns=['brand','model','Unnamed: 0'])

In [7]:
df.head()


,car_name,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [8]:

df.value_counts(['seller_type'])




seller_type     
Dealer              9539
Individual          5699
Trustmark Dealer     173
Name: count, dtype: int64

In [9]:
df.value_counts('fuel_type')

fuel_type
Petrol      7643
Diesel      7419
CNG          301
LPG           44
Electric       4
Name: count, dtype: int64

In [10]:
df.value_counts('transmission_type')

transmission_type
Manual       12225
Automatic     3186
Name: count, dtype: int64

In [11]:


### getting types of feature 

num_features = [features for features in df.columns if df[features].dtype not in ('object', 'str')]
print("numerical features are :", num_features, "\n")

cat_features = [features for features in df.columns if df[features].dtype in ('object', 'str')]
print("catigorical features are :", cat_features, "\n")

descrite_features = [features for features in num_features if len(df[features].unique()) <= 25]
print("descrite features are :", descrite_features, "\n")

continous_features = [features for features in num_features if features not in descrite_features]
print("continuous features are :", continous_features, "\n")

numerical features are : ['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats', 'selling_price'] 

catigorical features are : ['car_name', 'seller_type', 'fuel_type', 'transmission_type'] 

descrite features are : ['vehicle_age', 'seats'] 

continuous features are : ['km_driven', 'mileage', 'engine', 'max_power', 'selling_price'] 



##### train  test split

In [12]:
from sklearn.model_selection import train_test_split

X=df.drop('selling_price',axis=1)
y=df['selling_price']

## Feature Encoding and Scaling
**One Hot Encoding for Columns which had lesser unique values and not ordinal**
* One hot encoding is a process by which categorical variables are converted into a form that could be provided to ML algorithms to do a better job in prediction.

In [13]:
# first we need to transform the car_names in to numerical values for that we use lebal encoder

from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

X['car_name']=le.fit_transform(X['car_name'])


In [14]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

num_features_std=['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats'] ## selling price is not included in this one
cat_features_oh=['seller_type', 'fuel_type', 'transmission_type'] ## not including the car_name it is already included in lebal encoder

oh_encode=OneHotEncoder()
std_scaler=StandardScaler()

preprocessor=ColumnTransformer(
    [
        ("OneHotEncoder", oh_encode, cat_features_oh),
        ("StandardScaler", std_scaler, num_features_std)
        
    ],remainder ='passthrough'
)

In [15]:
X=preprocessor.fit_transform(X)

In [16]:
### train test split
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=45)



## Model Training And Model Selection

In [17]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [18]:
##Create a Function to Evaluate Model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [19]:
## Beginning Model Training
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
   
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 567049.6788
- Mean Absolute Error: 276600.1087
- R2 Score: 0.6246
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 439685.8642
- Mean Absolute Error: 264336.0319
- R2 Score: 0.6611


Lasso
Model performance for Training set
- Root Mean Squared Error: 567049.6821
- Mean Absolute Error: 276598.4422
- R2 Score: 0.6246
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 439685.1214
- Mean Absolute Error: 264333.7210
- R2 Score: 0.6611


Ridge
Model performance for Training set
- Root Mean Squared Error: 567050.1851
- Mean Absolute Error: 276566.7688
- R2 Score: 0.6246
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 439657.8109
- Mean Absolute Error: 264297.1772
- R2 Score: 0.6612


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 341465.2085
- Mean 